In [19]:
from __future__ import annotations
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from scipy.stats import ttest_1samp
from scipy.ndimage import gaussian_filter

import cmocean
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm
from typing import Iterable, Sequence

from dataclasses import dataclass
from pathlib import Path
from scipy.ndimage import gaussian_filter

from DataUtil import (
    build_experiments,
    DEFAULT_EXPERIMENTS
)

from ObsUtil import (
    OBS_REGISTRY,
    get_obs_file,
    list_obs_sources,
    obs_coverage,
)

In [20]:
@dataclass(frozen=True)
class SoilErrorMapPlotConfig:
    # data coords
    lat_name: str = "lat"
    lon_name: str = "lon"
    exp_dim: str = "exp"
    time_dim: str = "time"

    # variable names in file
    bias_var: str = "bias"
    spread_var: str = "model_std"   # "Spread" panel
    sigmask_var: str = "sig_mask"   # optional
    
    smooth: bool = False
    smooth_sigma_bias: float = 0.8     # ~1 grid cell
    smooth_sigma_spread: float = 0.6

    # map projection
    projection: ccrs.CRS = ccrs.Robinson()
    data_crs: ccrs.CRS = ccrs.PlateCarree()

    # figure layout
    figsize: tuple[float, float] = (11.5, 7.8)
    wspace: float = 0.08
    hspace: float = 0.10
    title_fontsize: int = 18

    # coastlines
    coast_lw: float = 1.2
    borders_lw: float = 0.6

    # Bias colorbar (diverging)
    bias_levels: tuple[float, ...] = (-8, -6, -4, -2, -1, 0, 1, 2, 4, 6, 8)
    bias_cmap: str = "BrBG_r"

    # Spread colorbar (sequential)
    spread_levels: tuple[float, ...] = (0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50)
    spread_cmap: str = "YlGnBu"

    # Caption control (default OFF)
    add_caption: bool = False
    caption_fontsize: int = 16

    # Stippling
    stipple: bool = False
    stipple_stride: int = 3
    stipple_size: float = 2.0
    stipple_color: str = "0.3"
    stipple_alpha: float = 0.6

    # Rendering tweaks
    shading: str = "nearest"     # "nearest" helps avoid mesh artifacts
    rasterized: bool = True      # good for pdf + reduces screen-door look


class SoilErrorMapPlotter:
    def __init__(self, cfg: SoilErrorMapPlotConfig):
        self.cfg = cfg

    # -------------------------
    # public API
    # -------------------------
    def plot_2x2(
        self,
        *,
        nc_file: str | Path,
        exp_left: str,
        exp_right: str,
        time_index: int = 0,
        obs_label: str | None = None,   # if None -> ds.attrs["obs"] if present
        caption: str | None = None,     # only used if cfg.add_caption=True
        out_png: str | Path | None = None,
        show: bool = True,
    ):
        ds = xr.open_dataset(nc_file)

        # normalize lon to 0..360 for clean global plotting
        ds = self._to_0_360(ds, lon_name=self.cfg.lon_name)

        lat = ds[self.cfg.lat_name]
        lon = ds[self.cfg.lon_name]

        # pick time
        if self.cfg.time_dim in ds.dims:
            ds_t = ds.isel({self.cfg.time_dim: time_index})
        else:
            ds_t = ds

        # obs label
        if obs_label is None:
            obs_label = ds.attrs.get("obs", "OBS")

        # extract fields
        bL, sL, pccL, rmseL, sigL = self._get_one(ds_t, exp_left)
        bR, sR, pccR, rmseR, sigR = self._get_one(ds_t, exp_right)

        # norms
        bias_levels = np.asarray(self.cfg.bias_levels, dtype=float)
        bias_norm = BoundaryNorm(bias_levels, ncolors=plt.get_cmap(self.cfg.bias_cmap).N, clip=False)

        spread_levels = np.asarray(self.cfg.spread_levels, dtype=float)
        spread_norm = BoundaryNorm(spread_levels, ncolors=plt.get_cmap(self.cfg.spread_cmap).N, clip=False)

        # ---- figure ----
        fig = plt.figure(figsize=self.cfg.figsize)
        gs = fig.add_gridspec(
            nrows=2, ncols=3,
            width_ratios=[1.0, 1.0, 0.06],
            wspace=self.cfg.wspace,
            hspace=self.cfg.hspace,
        )

        ax_bL = fig.add_subplot(gs[0, 0], projection=self.cfg.projection)
        ax_bR = fig.add_subplot(gs[0, 1], projection=self.cfg.projection)
        ax_sL = fig.add_subplot(gs[1, 0], projection=self.cfg.projection)
        ax_sR = fig.add_subplot(gs[1, 1], projection=self.cfg.projection)

        cax_bias = fig.add_subplot(gs[0, 2])
        cax_spre = fig.add_subplot(gs[1, 2])

        # ---- plot panels ----
        m_bL = self._plot_map(ax_bL, lon, lat, bL,
                              cmap=self.cfg.bias_cmap, norm=bias_norm, is_bias=True)
        m_bR = self._plot_map(ax_bR, lon, lat, bR,
                              cmap=self.cfg.bias_cmap, norm=bias_norm, is_bias=True)
        
        m_sL = self._plot_map(ax_sL, lon, lat, sL,
                              cmap=self.cfg.spread_cmap, norm=spread_norm, is_bias=False)
        m_sR = self._plot_map(ax_sR, lon, lat, sR,
                              cmap=self.cfg.spread_cmap, norm=spread_norm, is_bias=False)

        # titles
        ax_bL.set_title(f"{exp_left} - {obs_label} (Bias)", fontsize=self.cfg.title_fontsize, pad=10)
        ax_bR.set_title(f"{exp_right} - {obs_label} (Bias)", fontsize=self.cfg.title_fontsize, pad=10)
        ax_sL.set_title(f"{exp_left} (Spread)", fontsize=self.cfg.title_fontsize, pad=10)
        ax_sR.set_title(f"{exp_right} (Spread)", fontsize=self.cfg.title_fontsize, pad=10)

        # PCC/RMSE boxes on bias panels
        self._stats_box(ax_bL, pccL, rmseL)
        self._stats_box(ax_bR, pccR, rmseR)

        # stippling (ONLY over valid-data land points)
        if self.cfg.stipple:
            if sigL is not None:
                self._stipple(ax_bL, lon, lat, sigL, bL)
            if sigR is not None:
                self._stipple(ax_bR, lon, lat, sigR, bR)

        # ---- colorbars ----
        cb1 = plt.colorbar(m_bR, cax=cax_bias, orientation="vertical", extend="both")
        cb1.set_label("Bias (kg m$^{-2}$)", fontsize=self.cfg.title_fontsize)
        cb1.set_ticks(list(self.cfg.bias_levels))
        cb1.ax.tick_params(labelsize=self.cfg.title_fontsize - 2)

        cb2 = plt.colorbar(m_sR, cax=cax_spre, orientation="vertical", extend="max")
        cb2.set_label("Spread (kg m$^{-2}$)", fontsize=self.cfg.title_fontsize)
        cb2.set_ticks(list(self.cfg.spread_levels))
        cb2.ax.tick_params(labelsize=self.cfg.title_fontsize - 2)

        # caption: OFF by default; only if enabled explicitly
        if self.cfg.add_caption and caption:
            fig.text(0.5, 0.02, caption, ha="center", va="bottom", fontsize=self.cfg.caption_fontsize)

        if out_png is not None:
            out_png = Path(out_png)
            out_png.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(out_png, dpi=200, bbox_inches="tight")
        if show:
            plt.show()
        else:
            plt.close(fig)

        return fig

    # -------------------------
    # internals
    # -------------------------
    def _get_one(self, ds_t: xr.Dataset, exp: str):
        exp_dim = self.cfg.exp_dim
        if exp_dim in ds_t.dims:
            one = ds_t.sel({exp_dim: exp})
        else:
            one = ds_t

        bias = one[self.cfg.bias_var]
        spread = one[self.cfg.spread_var]

        pcc = float(one["pcc"].values) if "pcc" in one else np.nan
        rmse = float(one["rmse"].values) if "rmse" in one else np.nan

        sig = None
        if self.cfg.sigmask_var in one.data_vars:
            sig = one[self.cfg.sigmask_var]

        return bias, spread, pcc, rmse, sig

    def _plot_map(self, ax, lon, lat, field, *, cmap, norm, is_bias: bool):
        ax.set_global()
        ax.add_feature(cfeature.COASTLINE, linewidth=self.cfg.coast_lw)
        ax.add_feature(cfeature.BORDERS, linewidth=self.cfg.borders_lw)
    
        data = np.asarray(field.values)
    
        # ---- optional smoothing (visualization only) ----
        if self.cfg.smooth:
            sigma = (
                self.cfg.smooth_sigma_bias
                if is_bias
                else self.cfg.smooth_sigma_spread
            )
    
            # preserve NaNs during smoothing
            mask = np.isfinite(data)
            data_filled = np.where(mask, data, 0.0)
    
            smooth_data = gaussian_filter(data_filled, sigma=sigma, mode="nearest")
            smooth_mask = gaussian_filter(mask.astype(float), sigma=sigma, mode="nearest")
    
            with np.errstate(invalid="ignore"):
                data = np.where(smooth_mask > 0.5, smooth_data / smooth_mask, np.nan)
    
        m = ax.pcolormesh(
            lon, lat, data,
            transform=self.cfg.data_crs,
            cmap=plt.get_cmap(cmap),
            norm=norm,
            shading=self.cfg.shading,
            rasterized=self.cfg.rasterized,
        )
        return m


    def _stats_box(self, ax, pcc: float, rmse: float):
        txt = f"PCC: {pcc:.2f}\nRMSE: {rmse:.2f}"
        ax.text(
            0.98, 0.05, txt,
            transform=ax.transAxes,
            ha="right", va="bottom",
            fontsize=self.cfg.title_fontsize - 2,
            bbox=dict(boxstyle="square,pad=0.25", facecolor="white", edgecolor="black", linewidth=1.0),
        )

    def _stipple(self, ax, lon, lat, sigmask: xr.DataArray, field: xr.DataArray):
        """
        Stipple only where:
          - sigmask is True
          - field is finite (i.e., where you have valid land data)
        """
        sig = np.asarray(sigmask.values).astype(bool)
        val = np.isfinite(np.asarray(field.values))

        mask = sig & val
        if mask.ndim != 2 or not np.any(mask):
            return

        jj, ii = np.where(mask)

        stride = max(1, int(self.cfg.stipple_stride))
        jj = jj[::stride]
        ii = ii[::stride]

        LON, LAT = np.meshgrid(lon.values, lat.values)

        ax.scatter(
            LON[jj, ii], LAT[jj, ii],
            s=self.cfg.stipple_size,
            c=self.cfg.stipple_color,
            alpha=self.cfg.stipple_alpha,
            transform=self.cfg.data_crs,
            linewidths=0,
            zorder=5,
        )

    @staticmethod
    def _to_0_360(ds: xr.Dataset, lon_name: str = "lon") -> xr.Dataset:
        if lon_name not in ds.coords:
            return ds
        lon = ds[lon_name]
        if lon.min() < 0 or lon.max() > 360:
            ds = ds.assign_coords({lon_name: (lon % 360)}).sortby(lon_name)
        return ds

In [ ]:
if __name__ == "__main__":
    data_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    diag_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_output/som_error"

    fig_path = "./"
    os.makedirs(fig_path, exist_ok=True)

    exp_list = {
        "Jan2012": dict(
            models=["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"],
            period="201201-201203",
            init_time = "2012-01-01",
            season="Winter",
            init_month=1,
        ),
        "Jun2012": dict(
            models=["CTRL10-S1", "CAPT10-S1", "DART40-S1"],
            period="201206-201208",
            init_time = "2012-06-01",
            season="Summer",
            init_month=6,
        ),
    }

    group = "Jun2012"
    run = "fc"
    freq = "daily"

    # choose two experiments to compare
    if group == "Jan2012": 
        exp_target = ["CAPT10-S0", "DART40-S0"]
    else:
        exp_target = ["CAPT10-S1", "DART40-S1"]
        
    # obs + var
    obs_key = "ESA_CCI"
    var = "SOILWATER_10CM"

    # this MUST match the file you actually saved
    init_time = exp_list[group]["init_time"] # same as when you computed
    nc = (
        Path(diag_path)
        / f"soil_bias_{group}_{freq}_{run}_{obs_key}_{var}_{Path(pd.to_datetime(init_time).strftime('%Y%m%d'))}.nc"
    )
    # simpler (no Path tricks):
    # nc = f"{diag_path}/soil_bias_{group}_{freq}_{run}_{obs_key}_{var}_{pd.to_datetime(init_time):%Y%m%d}.nc"

    fig_name = f"{fig_path}/soil_diag_{group}_{var}_{pd.to_datetime(init_time):%Y%m%d}.png"

    caption = (
        "Soil moisture biases in January 2012 from ERA5-forced land model spin-up (CTRL) "
        "and EAMv3-DART after 1-month data assimilation."
    )

    bias_levels = (-10,-8, -6, -4, -2, 0, 2, 4, 6, 8, 10)
    spread_levels = (0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50)
    cmap_bias=cmocean.cm.curl
    cmap_std=cmocean.cm.deep
    stipple = False
    stipple_stride = 5
    stipple_size = 2.0
    stipple_color  = "0.3"
    stipple_alpha  = 0.6
    smooth=True
    smooth_sigma_bias=1.0
    smooth_sigma_spread=0.7
    stipple=False
    
    cfg = SoilErrorMapPlotConfig(
        bias_levels=bias_levels,
        spread_levels=spread_levels,
        bias_cmap = cmap_bias,
        spread_cmap = cmap_std,
        stipple = stipple,
        stipple_stride = stipple_stride,
        stipple_size = stipple_size,
        stipple_color  = stipple_color,
        stipple_alpha  =stipple_alpha,
        smooth=smooth,
        smooth_sigma_bias=smooth_sigma_bias,
        smooth_sigma_spread=smooth_sigma_spread,
    )
    plotter = SoilErrorMapPlotter(cfg)

    # your saved file is a single date -> time_index=0
    plotter.plot_2x2(
        nc_file=nc,
        exp_left=exp_target[0],
        exp_right=exp_target[1],
        time_index=0,
        obs_label=obs_key,
        caption=caption,
        out_png=fig_name,
        show=True,
    )

    print("Saved figure:", fig_name)
    